# PreRequisite

In [ ]:
import sys
# Define the path to the directory containing the module
module_dir = "../Main/RequiredFuntions"

# Append the directory to sys.path
sys.path.append(module_dir)

In [ ]:
# IMPORTS
import io
import cv2
import time
import json
import joblib
import hashlib
import imginfo
import datetime
import threading
import pandas as pd
import functions as fn
import dataTransfer as DT
import FingerBiometrics as FB
import InitializationPhase as IP
import EncryptionDecryption as ED
import accelerometerModelTraining as AT

# Initialization phase

In [ ]:
with open('../Main/ReceivedData/keys.json', 'r') as file:
    keys = json.load(file)

with open('../Main/ReceivedData/primenumber.json', 'r') as file:
    primenumber = json.load(file)

In [ ]:
primenumber = primenumber["primenumber"]

# USER - REGISTRATION PHASE

In [ ]:
# step 1
fingerprintImagePath = "../Main/RequiredData/Fingerprint/11.jpg"
accelerometerDataPath = "../Main/RequiredData/Accelerometer/m55s_training.csv"

In [ ]:
uniqueId_i = "RajeshDevice1"

imageHash = fn.hash_file(fingerprintImagePath)

accelerometerHash = fn.hash_file(accelerometerDataPath)

userNonce_i = fn.nonce_gen()


compute_i = hashlib.sha256((uniqueId_i 
                                + imageHash
                                + accelerometerHash
                                + str(userNonce_i)
                                + keys["key"]
                                ).encode()
                            ).hexdigest()

encryptedIdandImage = ED.symmetric_key_encryption(uniqueId_i, 
                                                  ED.read_image_as_bytes(fingerprintImagePath), 
                                                  keys["key"])

encryptedIDandAccelerometerData = ED.symmetric_key_encryption(uniqueId_i,
                                                              ED.read_image_as_bytes(accelerometerDataPath), 
                                                              keys["key"])

In [ ]:
FingerImageMetaData = imginfo.extract_metadata(fingerprintImagePath)
CSVFileMetaData = imginfo.extract_metadata(accelerometerDataPath)

encryptedFingerImageMetaData = ED.encryption(FingerImageMetaData, keys["gateway_public"].encode())
encryptedCSVFileMetaData = ED.encryption(CSVFileMetaData, keys["gateway_public"].encode())

In [ ]:
userDataStep1 = {
    "encryptedIdandImage" : encryptedIdandImage,
    "encryptedIDandAccelerometerData" : encryptedIDandAccelerometerData,
    "imageHash" : imageHash,
    "accelerometerHash" : accelerometerHash,
    "userNonce" : userNonce_i,
    "compute" : compute_i,
    "FingerImageMetaData" : encryptedFingerImageMetaData,
    "CSVFileMetaData" : encryptedCSVFileMetaData
}


In [ ]:
# Create and start threads
thread1 = threading.Thread(target=DT.receive)  # Start the receive function
thread2 = threading.Thread(target=DT.send, args=(userDataStep1,))  # Start the send function with arguments

thread1.start()
time.sleep(2)  # Ensure the server starts before sending
thread2.start()

# Wait for threads to complete
thread1.join()
thread2.join()

print("Data transfer and storage complete.")

In [ ]:
# step 2
print("Starting Step 2")
userStep1Data = fn.read_json_file("../Main/ReceivedData/received_data.json")

In [ ]:
decryptedIdandImage = ED.symmetric_key_decryption(userStep1Data["encryptedIdandImage"], 
                                                  keys["key"])

decryptedIDandAccelerometerData = ED.symmetric_key_decryption(userStep1Data["encryptedIDandAccelerometerData"], 
                                                              keys["key"])

print(f"both the above decryptions gave the same output unique id : {decryptedIdandImage[0] == decryptedIDandAccelerometerData[0]}")

print("image hash validated : ", hashlib.sha256(decryptedIdandImage[1]).hexdigest() == userStep1Data["imageHash"])

In [ ]:
validationCompute_i = hashlib.sha256((decryptedIdandImage[0].decode() + 
                                      hashlib.sha256(decryptedIdandImage[1]).hexdigest() + 
                                      hashlib.sha256(decryptedIDandAccelerometerData[1]).hexdigest() + 
                                      str(userStep1Data["userNonce"]) +
                                      keys["key"]).encode()).hexdigest()

In [ ]:
if(validationCompute_i == userStep1Data["compute"]): print("validation for compute status : \"success\"")
else: print("validation \"failed\" ")

In [ ]:
baseAccelerometerData = "../Main/RequiredData/Accelerometer/baseTrainingData.csv"
gatewayUniqueId_i = "rajeshGateway"

In [ ]:
baseDF = pd.read_csv(baseAccelerometerData)

In [ ]:
df = pd.read_csv(io.StringIO(decryptedIDandAccelerometerData[1].decode('UTF-8')))
df = df.dropna()
df = df.drop(columns=["Var1_Timestamp", "Var2_Timestamp", "Var3_Timestamp"])
df["user"] = decryptedIdandImage[0].decode()
df.head()

In [ ]:
Combined_training_data = pd.concat([baseDF, df], ignore_index=True)
Combined_training_data.to_csv(baseAccelerometerData, index=False)

In [ ]:
TrainingThread = threading.Thread(target=AT.AccelerometerTraining, args=(Combined_training_data,))
TrainingThread.start()

In [ ]:
# checking the liveness of the image
fingerprintModel = joblib.load('./Model/Fingerprint/naive_bayes_fingerprint_model.pkl')
fingerprintPrediction = FB.predict_fingerprint(decryptedIdandImage[1], (200,200), fingerprintModel)
print(f"The fingerprint is: {fingerprintPrediction}")

    #   should add a break statement if image is fake

In [ ]:
cropped_image = FB.IsolateFingerImage(decryptedIdandImage[1])

In [ ]:
# saving image to bytes
_, buffer = cv2.imencode('.jpg', cropped_image)
fingerImage = buffer.tobytes()

In [ ]:
hashedGatewayID = hashlib.sha256(gatewayUniqueId_i.encode()).hexdigest()
hashedUniqueId_i = hashlib.sha256(uniqueId_i.encode()).hexdigest()

In [ ]:
# encrypt image using pubic key (gateway)
_, buffer = cv2.imencode('.jpg', cropped_image)
encryptedImage = ED.encryption(buffer.tobytes(), keys["gateway_public"].encode())

In [ ]:
#generate nounce in gateway
gatewayNonce_i = fn.nonce_gen()

Secret_i = int(gatewayNonce_i) ^ int(userStep1Data["userNonce"])

In [ ]:
Secret_i

In [ ]:
# shares generated by gateway
userGeneratedRandomNumber = fn.nonce_gen()
userSharesGeneratedByGateway = (Secret_i + (2 * userGeneratedRandomNumber)) % primenumber

In [ ]:
secretIntegrity_i_partial = hashlib.sha256((str(Secret_i) + decryptedIdandImage[0].decode()).encode()).hexdigest()

In [ ]:
dict_storage = {
    gatewayUniqueId_i: {
            uniqueId_i: {
               "partialSecretIntegrityUser": secretIntegrity_i_partial,
               "gatewayShare": userSharesGeneratedByGateway
            }
        }
    }

dict_storage

In [ ]:
with open(f'./ReceivedData/datastore.json', 'w') as json_file:
        json.dump(dict_storage, json_file, indent=4)

In [ ]:
secretIntegrity_i = hashlib.sha256((secretIntegrity_i_partial + str(userStep1Data["userNonce"])).encode()).hexdigest()

In [ ]:
# temporal identity to encrypt R and gatwayNonce

temporalIdentity_i = hashlib.sha256((decryptedIdandImage[0].decode() + keys["user_public"] + str(userStep1Data["userNonce"])).encode()).hexdigest()

In [ ]:
encryptedRandomNumber = fn.xor_strings(str(userGeneratedRandomNumber), temporalIdentity_i)
encryptedRandomNumber = fn.xor_strings(encryptedRandomNumber, keys["key"])

In [ ]:
encryptedGatewayNonce = fn.xor_strings(str(gatewayNonce_i), temporalIdentity_i)
encryptedGatewayNonce = fn.xor_strings(encryptedGatewayNonce, keys["key"])

In [ ]:
gatewaySecondNonce_i = fn.nonce_gen()

userTemporalIdentityGateway = hashlib.sha256((gatewayUniqueId_i + keys["gateway_public"] + str(gatewaySecondNonce_i)).encode()).hexdigest()

compute_G = hashlib.sha256((
    userTemporalIdentityGateway + 
    secretIntegrity_i + 
    encryptedRandomNumber + 
    encryptedGatewayNonce + 
    str(gatewaySecondNonce_i)).encode()).hexdigest()

In [ ]:
userGatewayJson = {
    'secretIntegrity' : secretIntegrity_i,
    'encryptedRandomNumber' : encryptedRandomNumber,
    'encryptedGatewayNonce' : encryptedGatewayNonce,
    'gatwayNonce' : gatewaySecondNonce_i,
    'computeG' : compute_G
}

In [ ]:
# Create and start threads
thread1 = threading.Thread(target=DT.receive, args=('received_data_gateway.json',))  # Start the receive function
thread2 = threading.Thread(target=DT.send, args=(userGatewayJson,))  # Start the send function with arguments

thread1.start()
time.sleep(2)  # Ensure the server starts before sending
thread2.start()

# Wait for threads to complete
thread1.join()
thread2.join()

print("Data transfer and storage complete.")

In [ ]:
TrainingThread.join()

    # step 3

In [ ]:
# step 2
print("Starting Step 3")
gatewayJsonData = fn.read_json_file("../Main/ReceivedData/received_data_gateway.json")

In [ ]:
#compute temporal identity user
temporalIdentity_i_s3 = hashlib.sha256((uniqueId_i + keys["user_public"] + str(userNonce_i)).encode()).hexdigest()

#compute temporal indentity gateway
temporalIdentity_g_s3 = hashlib.sha256((gatewayUniqueId_i + keys["gateway_public"] + str(gatewayJsonData['gatwayNonce'])).encode()).hexdigest()

In [ ]:
validateComputeG = hashlib.sha256((temporalIdentity_g_s3 + 
                    gatewayJsonData['secretIntegrity'] + 
                    str(gatewayJsonData['encryptedRandomNumber']) + 
                    str(gatewayJsonData['encryptedGatewayNonce']) + 
                    str(gatewayJsonData['gatwayNonce'])).encode()).hexdigest()

In [ ]:
validateComputeG == gatewayJsonData["computeG"]

In [ ]:
# unencrypt randomnumber and gateway nonce
userGatewayRandomNumber = fn.xor_strings(gatewayJsonData['encryptedRandomNumber'], temporalIdentity_i_s3)
userGatewayRandomNumber = fn.xor_strings(userGatewayRandomNumber, keys["key"])

In [ ]:
userGatewayGeneratedNonce = fn.xor_strings(gatewayJsonData['encryptedGatewayNonce'], temporalIdentity_i_s3)
userGatewayGeneratedNonce = fn.xor_strings(userGatewayGeneratedNonce, keys["key"])

In [ ]:
userSecretComputed = userNonce_i ^ int(userGatewayGeneratedNonce)
userShare = (userSecretComputed + int(userGatewayRandomNumber)) % primenumber

In [ ]:
userValidationSecretIntegrity = hashlib.sha256((str(userSecretComputed) + uniqueId_i).encode()).hexdigest()
userValidationSecretIntegrity = hashlib.sha256((userValidationSecretIntegrity + str(userNonce_i)).encode()).hexdigest()

In [ ]:
print(f"Secret Integrity validation : {userValidationSecretIntegrity == secretIntegrity_i}")

In [ ]:
with open(f'./ReceivedData/datastore.json', 'r') as json_file:
        datastore = json.load(json_file)

In [ ]:
datastore[gatewayUniqueId_i][uniqueId_i]["userShare"] = userShare
datastore[gatewayUniqueId_i][uniqueId_i]["partialSecretIntegrityGateway"] = hashlib.sha256((str(userSecretComputed) + gatewayUniqueId_i).encode()).hexdigest()

In [ ]:
with open(f'./ReceivedData/datastore.json', 'w') as json_file:
        json.dump(datastore, json_file, indent=4)